In [1]:
import numpy as np
import pandas as pd
import pandas_gbq

In [2]:
import pandas as pd

# 1. Charger les tables principales
catnat = pd.read_csv("catnat_staging.csv", low_memory=False)
pprn = pd.read_csv("pprn_garspar_staging.csv", low_memory=False)
pprm = pd.read_csv("pprm_gaspar_staging.csv", low_memory=False)
pprt = pd.read_csv("pprt_gaspar_staging.csv", low_memory=False)


# 2. Classifier et compter les catastrophes naturelles (inondation, tempête, séisme)
def classify_risk(lib):
  if pd.isna(lib):
    return None
  lib_lower = str(lib).lower()
  if "inondation" in lib_lower or "coulees de boue" in lib_lower:
    return "nb_innondation"
  elif "tempete" in lib_lower:
    return "nb_tempete"
  elif "sismique" in lib_lower or "seisme" in lib_lower:
    return "nb_seisme"
  return None


catnat["risk_type"] = catnat["lib_risque_jo"].apply(classify_risk)

# Pivoter pour obtenir les colonnes de comptage
catnat_counts = (
    catnat.dropna(subset=["risk_type"])
    .pivot_table(
        index="code_commune",
        columns="risk_type",
        values="id_gaspar",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)

# Récupérer la dernière date de publication au journal officiel
catnat_jo = (
    catnat.groupby("code_commune")["date_publication_jo"]
    .max()
    .reset_index()
)
catnat_jo = catnat_jo.rename(
    columns={"date_publication_jo": "date de publication au journal officelles"}
)

# 3. Traiter les plans de prévention (PPRn, PPRm, PPRt)
pprn["clean_code_commune"] = pprn["code_commune"].fillna(
    pprn["code_commune_final"]
)
pprn_sub = pprn[pprn["clean_code_commune"].notna()][
    [
        "clean_code_commune",
        "nom_commune",
        "nom_region",
        "nom_departement",
        "code_insee_region",
        "code_insee_departement",
        "libelle_etat",
    ]
].copy()
pprn_sub = pprn_sub.rename(
    columns={
        "clean_code_commune": "code_commune",
        "libelle_etat": "plan de prévention des risques naturel",
    }
)

# Répéter ou fusionner progressivement selon vos clés géographiques et communales
# (Exemple de fusion séquentielle sur code_commune)

FileNotFoundError: [Errno 2] No such file or directory: 'catnat_staging.csv'

In [3]:
import numpy as np
import pandas as pd

# 1. Charger les fichiers source
catnat = pd.read_csv('catnat_staging.csv', low_memory=False)
pprn = pd.read_csv('pprn_garspar_staging.csv', low_memory=False)
pprm = pd.read_csv('pprm_gaspar_staging.csv', low_memory=False)
pprt = pd.read_csv('pprt_gaspar_staging.csv', low_memory=False)

# 2. Traitement et agrégation des Catastrophes Naturelles (CatNat)
def classify_risk(lib):
  if pd.isna(lib):
    return None
  lib_lower = str(lib).lower()
  if 'inondation' in lib_lower or 'coulees de boue' in lib_lower:
    return 'nb_innondation'
  elif 'tempete' in lib_lower:
    return 'nb_tempete'
  elif 'sismique' in lib_lower or 'seisme' in lib_lower:
    return 'nb_seisme'
  return None

catnat['risk_type'] = catnat['lib_risque_jo'].apply(classify_risk)
catnat_counts = (
    catnat.dropna(subset=['risk_type'])
    .pivot_table(
        index='code_commune',
        columns='risk_type',
        values='id_gaspar',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

for col in ['nb_innondation', 'nb_tempete', 'nb_seisme']:
  if col not in catnat_counts.columns:
    catnat_counts[col] = 0

catnat_jo = (
    catnat.groupby('code_commune')['date_publication_jo']
    .max()
    .reset_index()
)
catnat_jo = catnat_jo.rename(
    columns={'date_publication_jo': 'date de publication au journal officelles'}
)

catnat_merged = pd.merge(
    catnat_counts, catnat_jo, on='code_commune', how='outer'
)

# 3. Traitement des Plans de Prévention des Risques Naturels (PPRn)
pprn['code_commune_clean'] = pprn['code_commune'].fillna(
    pprn['code_commune_final']
)
pprn_agg = (
    pprn.dropna(subset=['code_commune_clean'])
    .groupby('code_commune_clean')
    .agg({
        'nom_commune': 'first',
        'nom_region': 'first',
        'nom_departement': 'first',
        'code_insee_region': 'first',
        'code_insee_departement': 'first',
        'libelle_etat': lambda x: 'Oui' if any(pd.notna(x)) else 'Non',
    })
    .reset_index()
)

pprn_agg = pprn_agg.rename(
    columns={
        'code_commune_clean': 'code commune',
        'code_insee_region': 'code région',
        'code_insee_departement': 'code département',
        'nom_commune': 'nom commune',
        'nom_region': 'nom region',
        'nom_departement': 'nom départment',
        'libelle_etat': 'plan de prévention des risques naturel',
    }
)

# 4. Traitement des Risques Miniers (PPRm)
pprm['code_commune_clean'] = (
    pprm['CODE INSEE COMMUNE'].astype(str).str.zfill(5)
)
pprm_agg = (
    pprm.groupby('code_commune_clean')
    .agg({
        'LIBELLE ETAT': lambda x: (
            'Oui' if any(pd.notna(x) & (x != 'none')) else 'Non'
        )
    })
    .reset_index()
    .rename(
        columns={
            'code_commune_clean': 'code commune',
            'LIBELLE ETAT': 'plan de prévention des risques miniers',
        }
    )
)

# 5. Traitement des Risques Technologiques (PPRt)
pprt['code_commune_clean'] = (
    pprt['CODE INSEE COMMUNE'].astype(str).str.zfill(5)
)
pprt_agg = (
    pprt.groupby('code_commune_clean')
    .agg({
        'LIBELLE ETAT': lambda x: (
            'Oui' if any(pd.notna(x) & (x != 'none')) else 'Non'
        )
    })
    .reset_index()
    .rename(
        columns={
            'code_commune_clean': 'code commune',
            'LIBELLE ETAT': 'plan de prévention des risques technologiques',
        }
    )
)

# 6. Fusion finale de toutes les tables
catnat_merged['code commune'] = (
    catnat_merged['code_commune'].astype(str).str.zfill(5)
)

final_df = pprn_agg[
    [
        'code commune',
        'code région',
        'code département',
        'nom commune',
        'nom region',
        'nom départment',
    ]
].copy()

final_df = pd.merge(
    final_df,
    catnat_merged[[
        'code commune',
        'nb_innondation',
        'nb_tempete',
        'nb_seisme',
        'date de publication au journal officelles',
    ]],
    on='code commune',
    how='left',
)
final_df = pd.merge(
    final_df,
    pprn_agg[['code commune', 'plan de prévention des risques naturel']],
    on='code commune',
    how='left',
)
final_df = pd.merge(final_df, pprm_agg, on='code commune', how='left')
final_df = pd.merge(final_df, pprt_agg, on='code commune', how='left')

# Colonne swi_unif_mens (à relier avec votre table SWI si disponible)
final_df['swi_unif_mens'] = np.nan

# Ordonnancement exact des colonnes demandées
cols_order = [
    'swi_unif_mens',
    'code commune',
    'code région',
    'code département',
    'nom commune',
    'nom region',
    'nom départment',
    'nb_innondation',
    'nb_tempete',
    'nb_seisme',
    'plan de prévention des risques naturel',
    'plan de prévention des risques miniers',
    'plan de prévention des risques technologiques',
    'date de publication au journal officelles',
]

for col in cols_order:
  if col not in final_df.columns:
    final_df[col] = np.nan

final_df = final_df[cols_order]

# Affichage des premières lignes
print(final_df.head())

# Optionnel : Sauvegarde en CSV
final_df.to_csv('table_finale_risques.csv', index=False, encoding='utf-8-sig')

FileNotFoundError: [Errno 2] No such file or directory: 'catnat_staging.csv'

In [ ]:
final_df

,swi_unif_mens,code commune,code région,code département,nom commune,nom region,nom départment,nb_innondation,nb_tempete,nb_seisme,plan de prévention des risques naturel,plan de prévention des risques miniers,plan de prévention des risques technologiques,date de publication au journal officelles
0,NaN,01004,84.0,01,amberieu-en-bugey,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
1,NaN,01007,84.0,01,ambronay,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1990-03-23
2,NaN,01010,84.0,01,anglefort,auvergne-rhone-alpes,ain,NaN,NaN,NaN,Oui,NaN,NaN,NaN
3,NaN,01016,84.0,01,arbigny,auvergne-rhone-alpes,ain,4.0,0.0,0.0,Oui,NaN,NaN,1990-12-19
4,NaN,01017,84.0,01,argis,auvergne-rhone-alpes,ain,2.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14240,NaN,97615,6.0,976,pamandzi,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14241,NaN,97616,6.0,976,sada,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14242,NaN,97617,6.0,976,tsingoni,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14243,NaN,97701,7.0,977,saint-barthelemy,saint-barthelemy,saint-barthelemy,NaN,NaN,NaN,Oui,NaN,NaN,NaN


In [ ]:
import numpy as np
import pandas as pd

# 1. Charger tous les fichiers source (y compris la table SWI)
catnat = pd.read_csv("catnat_staging.csv", low_memory=False)
pprn = pd.read_csv("pprn_garspar_staging.csv", low_memory=False)
pprm = pd.read_csv("pprm_gaspar_staging.csv", low_memory=False)
pprt = pd.read_csv("pprt_gaspar_staging.csv", low_memory=False)
swi_df = pd.read_csv("swi_par_commune.csv", low_memory=False)  # Ajustez le nom si besoin

# 2. Traitement de la table SWI (swi_unif_mens)
# Si vous avez plusieurs lignes par commune (ex: par date), vous pouvez prendre la moyenne ou la dernière valeur
swi_df["code_commune_clean"] = (
    swi_df["code_commune"].astype(str).str.zfill(5)
)

# Exemple : calcul de la moyenne du taux d'humidité par commune (ou .last() / .first() selon votre besoin)
swi_agg = (
    swi_df.groupby("code_commune_clean")["SWI_UNIF_MENS"]
    .mean()
    .reset_index()
)
swi_agg = swi_agg.rename(
    columns={
        "code_commune_clean": "code commune",
        "SWI_UNIF_MENS": "swi_unif_mens",
    }
)


# 3. Traitement et agrégation des Catastrophes Naturelles (CatNat)
def classify_risk(lib):
  if pd.isna(lib):
    return None
  lib_lower = str(lib).lower()
  if "inondation" in lib_lower or "coulees de boue" in lib_lower:
    return "nb_innondation"
  elif "tempete" in lib_lower:
    return "nb_tempete"
  elif "sismique" in lib_lower or "seisme" in lib_lower:
    return "nb_seisme"
  return None


catnat["risk_type"] = catnat["lib_risque_jo"].apply(classify_risk)
catnat_counts = (
    catnat.dropna(subset=["risk_type"])
    .pivot_table(
        index="code_commune",
        columns="risk_type",
        values="id_gaspar",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)

for col in ["nb_innondation", "nb_tempete", "nb_seisme"]:
  if col not in catnat_counts.columns:
    catnat_counts[col] = 0

catnat_jo = (
    catnat.groupby("code_commune")["date_publication_jo"]
    .max()
    .reset_index()
)
catnat_jo = catnat_jo.rename(
    columns={"date_publication_jo": "date de publication au journal officelles"}
)

catnat_merged = pd.merge(
    catnat_counts, catnat_jo, on="code_commune", how="outer"
)

# 4. Traitement des Plans de Prévention des Risques Naturels (PPRn)
pprn["code_commune_clean"] = pprn["code_commune"].fillna(
    pprn["code_commune_final"]
)
pprn_agg = (
    pprn.dropna(subset=["code_commune_clean"])
    .groupby("code_commune_clean")
    .agg({
        "nom_commune": "first",
        "nom_region": "first",
        "nom_departement": "first",
        "code_insee_region": "first",
        "code_insee_departement": "first",
        "libelle_etat": lambda x: "Oui" if any(pd.notna(x)) else "Non",
    })
    .reset_index()
)

pprn_agg = pprn_agg.rename(
    columns={
        "code_commune_clean": "code commune",
        "code_insee_region": "code région",
        "code_insee_departement": "code département",
        "nom_commune": "nom commune",
        "nom_region": "nom region",
        "nom_departement": "nom départment",
        "libelle_etat": "plan de prévention des risques naturel",
    }
)

# 5. Traitement des Risques Miniers (PPRm)
pprm["code_commune_clean"] = (
    pprm["CODE INSEE COMMUNE"].astype(str).str.zfill(5)
)
pprm_agg = (
    pprm.groupby("code_commune_clean")
    .agg({
        "LIBELLE ETAT": lambda x: (
            "Oui" if any(pd.notna(x) & (x != "none")) else "Non"
        )
    })
    .reset_index()
    .rename(
        columns={
            "code_commune_clean": "code commune",
            "LIBELLE ETAT": "plan de prévention des risques miniers",
        }
    )
)

# 6. Traitement des Risques Technologiques (PPRt)
pprt["code_commune_clean"] = (
    pprt["CODE INSEE COMMUNE"].astype(str).str.zfill(5)
)
pprt_agg = (
    pprt.groupby("code_commune_clean")
    .agg({
        "LIBELLE ETAT": lambda x: (
            "Oui" if any(pd.notna(x) & (x != "none")) else "Non"
        )
    })
    .reset_index()
    .rename(
        columns={
            "code_commune_clean": "code commune",
            "LIBELLE ETAT": "plan de prévention des risques technologiques",
        }
    )
)

# 7. Fusion finale de toutes les tables (en incluant swi_agg)
catnat_merged["code commune"] = (
    catnat_merged["code_commune"].astype(str).str.zfill(5)
)

final_df = pprn_agg[
    [
        "code commune",
        "code région",
        "code département",
        "nom commune",
        "nom region",
        "nom départment",
    ]
].copy()

# Intégration du taux d'humidité SWI
final_df = pd.merge(final_df, swi_agg, on="code commune", how="left")

final_df = pd.merge(
    final_df,
    catnat_merged[[
        "code commune",
        "nb_innondation",
        "nb_tempete",
        "nb_seisme",
        "date de publication au journal officelles",
    ]],
    on="code commune",
    how="left",
)
final_df = pd.merge(
    final_df,
    pprn_agg[["code commune", "plan de prévention des risques naturel"]],
    on="code commune",
    how="left",
)
final_df = pd.merge(final_df, pprm_agg, on="code commune", how="left")
final_df = pd.merge(final_df, pprt_agg, on="code commune", how="left")

# Ordonnancement exact des colonnes demandées
cols_order = [
    "swi_unif_mens",
    "code commune",
    "code région",
    "code département",
    "nom commune",
    "nom region",
    "nom départment",
    "nb_innondation",
    "nb_tempete",
    "nb_seisme",
    "plan de prévention des risques naturel",
    "plan de prévention des risques miniers",
    "plan de prévention des risques technologiques",
    "date de publication au journal officelles",
]

for col in cols_order:
  if col not in final_df.columns:
    final_df[col] = np.nan

final_df = final_df[cols_order]

# Affichage et export
print(final_df.head())
final_df.to_csv("table_finale_avec_swi.csv", index=False, encoding="utf-8-sig")

   swi_unif_mens code commune  code région code département  \
0            NaN        01004         84.0               01   
1            NaN        01007         84.0               01   
2       0.852557        01010         84.0               01   
3            NaN        01016         84.0               01   
4            NaN        01017         84.0               01   

         nom commune            nom region nom départment  nb_innondation  \
0  amberieu-en-bugey  auvergne-rhone-alpes            ain             3.0   
1           ambronay  auvergne-rhone-alpes            ain             3.0   
2          anglefort  auvergne-rhone-alpes            ain             NaN   
3            arbigny  auvergne-rhone-alpes            ain             4.0   
4              argis  auvergne-rhone-alpes            ain             2.0   

   nb_tempete  nb_seisme plan de prévention des risques naturel  \
0         0.0        0.0                                    Oui   
1         0.0        0.0

In [ ]:
final_df

,swi_unif_mens,code commune,code région,code département,nom commune,nom region,nom départment,nb_innondation,nb_tempete,nb_seisme,plan de prévention des risques naturel,plan de prévention des risques miniers,plan de prévention des risques technologiques,date de publication au journal officelles
0,NaN,01004,84.0,01,amberieu-en-bugey,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
1,NaN,01007,84.0,01,ambronay,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1990-03-23
2,0.852557,01010,84.0,01,anglefort,auvergne-rhone-alpes,ain,NaN,NaN,NaN,Oui,NaN,NaN,NaN
3,NaN,01016,84.0,01,arbigny,auvergne-rhone-alpes,ain,4.0,0.0,0.0,Oui,NaN,NaN,1990-12-19
4,NaN,01017,84.0,01,argis,auvergne-rhone-alpes,ain,2.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14240,NaN,97615,6.0,976,pamandzi,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14241,NaN,97616,6.0,976,sada,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14242,NaN,97617,6.0,976,tsingoni,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14243,NaN,97701,7.0,977,saint-barthelemy,saint-barthelemy,saint-barthelemy,NaN,NaN,NaN,Oui,NaN,NaN,NaN


In [ ]:
# Afficher le nombre de valeurs nulles par colonne
null_counts = final_df.isnull().sum()
print("Nombre de valeurs nulles par colonne :")
print(null_counts)

# Pour afficher également le pourcentage de valeurs nulles (optionnel mais très utile)
null_percent = (final_df.isnull().mean() * 100).round(2)
null_summary = pd.DataFrame(
    {"Nombre_Nulls": null_counts, "Pourcentage_Nulls (%)": null_percent}
)
print("\nRésumé détaillé des valeurs nulles :")
print(null_summary)

Nombre de valeurs nulles par colonne :
swi_unif_mens                                    10905
code commune                                         0
code région                                        806
code département                                     0
nom commune                                         33
nom region                                         806
nom départment                                       0
nb_innondation                                    3177
nb_tempete                                        3177
nb_seisme                                         3177
plan de prévention des risques naturel               0
plan de prévention des risques miniers           14121
plan de prévention des risques technologiques    13690
date de publication au journal officelles         2906
dtype: int64

Résumé détaillé des valeurs nulles :
                                               Nombre_Nulls  \
swi_unif_mens                                         10905   
code commune  

In [ ]:
final_df

,swi_unif_mens,code commune,code région,code département,nom commune,nom region,nom départment,nb_innondation,nb_tempete,nb_seisme,plan de prévention des risques naturel,plan de prévention des risques miniers,plan de prévention des risques technologiques,date de publication au journal officelles
0,NaN,01004,84.0,01,amberieu-en-bugey,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
1,NaN,01007,84.0,01,ambronay,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1990-03-23
2,0.852557,01010,84.0,01,anglefort,auvergne-rhone-alpes,ain,NaN,NaN,NaN,Oui,NaN,NaN,NaN
3,NaN,01016,84.0,01,arbigny,auvergne-rhone-alpes,ain,4.0,0.0,0.0,Oui,NaN,NaN,1990-12-19
4,NaN,01017,84.0,01,argis,auvergne-rhone-alpes,ain,2.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14240,NaN,97615,6.0,976,pamandzi,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14241,NaN,97616,6.0,976,sada,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14242,NaN,97617,6.0,976,tsingoni,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14243,NaN,97701,7.0,977,saint-barthelemy,saint-barthelemy,saint-barthelemy,NaN,NaN,NaN,Oui,NaN,NaN,NaN


In [ ]:
import numpy as np
import pandas as pd
import pandas_gbq

# 1. Charger tous les fichiers source (y compris la table SWI)
catnat = pd.read_csv("catnat_staging.csv", low_memory=False)
pprn = pd.read_csv("pprn_garspar_staging.csv", low_memory=False)
pprm = pd.read_csv("pprm_gaspar_staging.csv", low_memory=False)
pprt = pd.read_csv("pprt_gaspar_staging.csv", low_memory=False)
swi_df = pd.read_csv("swi_par_commune.csv", low_memory=False)

# 2. Traitement de la table SWI (swi_unif_mens)
swi_df["code_commune_clean"] = (
    swi_df["code_commune"].astype(str).str.zfill(5)
)
swi_agg = (
    swi_df.groupby("code_commune_clean")["SWI_UNIF_MENS"]
    .mean()
    .reset_index()
)
swi_agg = swi_agg.rename(
    columns={
        "code_commune_clean": "code commune",
        "SWI_UNIF_MENS": "swi_unif_mens",
    }
)


# 3. Traitement et agrégation des Catastrophes Naturelles (CatNat)
def classify_risk(lib):
  if pd.isna(lib):
    return None
  lib_lower = str(lib).lower()
  if "inondation" in lib_lower or "coulees de boue" in lib_lower:
    return "nb_innondation"
  elif "tempete" in lib_lower:
    return "nb_tempete"
  elif "sismique" in lib_lower or "seisme" in lib_lower:
    return "nb_seisme"
  return None


catnat["risk_type"] = catnat["lib_risque_jo"].apply(classify_risk)
catnat_counts = (
    catnat.dropna(subset=["risk_type"])
    .pivot_table(
        index="code_commune",
        columns="risk_type",
        values="id_gaspar",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)

for col in ["nb_innondation", "nb_tempete", "nb_seisme"]:
  if col not in catnat_counts.columns:
    catnat_counts[col] = 0

catnat_jo = (
    catnat.groupby("code_commune")["date_publication_jo"]
    .max()
    .reset_index()
)
catnat_jo = catnat_jo.rename(
    columns={"date_publication_jo": "date de publication au journal officelles"}
)

catnat_merged = pd.merge(
    catnat_counts, catnat_jo, on="code_commune", how="outer"
)

# 4. Traitement des Plans de Prévention des Risques Naturels (PPRn)
pprn["code_commune_clean"] = pprn["code_commune"].fillna(
    pprn["code_commune_final"]
)
pprn_agg = (
    pprn.dropna(subset=["code_commune_clean"])
    .groupby("code_commune_clean")
    .agg({
        "nom_commune": "first",
        "nom_region": "first",
        "nom_departement": "first",
        "code_insee_region": "first",
        "code_insee_departement": "first",
        "libelle_etat": lambda x: "Oui" if any(pd.notna(x)) else "Non",
    })
    .reset_index()
)

pprn_agg = pprn_agg.rename(
    columns={
        "code_commune_clean": "code commune",
        "code_insee_region": "code région",
        "code_insee_departement": "code département",
        "nom_commune": "nom commune",
        "nom_region": "nom region",
        "nom_departement": "nom départment",
        "libelle_etat": "plan de prévention des risques naturel",
    }
)

# 5. Traitement des Risques Miniers (PPRm)
pprm["code_commune_clean"] = (
    pprm["CODE INSEE COMMUNE"].astype(str).str.zfill(5)
)
pprm_agg = (
    pprm.groupby("code_commune_clean")
    .agg({
        "LIBELLE ETAT": lambda x: (
            "Oui" if any(pd.notna(x) & (x != "none")) else "Non"
        )
    })
    .reset_index()
    .rename(
        columns={
            "code_commune_clean": "code commune",
            "LIBELLE ETAT": "plan de prévention des risques miniers",
        }
    )
)

# 6. Traitement des Risques Technologiques (PPRt)
pprt["code_commune_clean"] = (
    pprt["CODE INSEE COMMUNE"].astype(str).str.zfill(5)
)
pprt_agg = (
    pprt.groupby("code_commune_clean")
    .agg({
        "LIBELLE ETAT": lambda x: (
            "Oui" if any(pd.notna(x) & (x != "none")) else "Non"
        )
    })
    .reset_index()
    .rename(
        columns={
            "code_commune_clean": "code commune",
            "LIBELLE ETAT": "plan de prévention des risques technologiques",
        }
    )
)

# 7. Fusion finale de toutes les tables
catnat_merged["code commune"] = (
    catnat_merged["code_commune"].astype(str).str.zfill(5)
)

final_df = pprn_agg[
    [
        "code commune",
        "code région",
        "code département",
        "nom commune",
        "nom region",
        "nom départment",
    ]
].copy()

final_df = pd.merge(final_df, swi_agg, on="code commune", how="left")
final_df = pd.merge(
    final_df,
    catnat_merged[[
        "code commune",
        "nb_innondation",
        "nb_tempete",
        "nb_seisme",
        "date de publication au journal officelles",
    ]],
    on="code commune",
    how="left",
)
final_df = pd.merge(
    final_df,
    pprn_agg[["code commune", "plan de prévention des risques naturel"]],
    on="code commune",
    how="left",
)
final_df = pd.merge(final_df, pprm_agg, on="code commune", how="left")
final_df = pd.merge(final_df, pprt_agg, on="code commune", how="left")

# Ordonnancement exact des colonnes demandées
cols_order = [
    "swi_unif_mens",
    "code commune",
    "code région",
    "code département",
    "nom commune",
    "nom region",
    "nom départment",
    "nb_innondation",
    "nb_tempete",
    "nb_seisme",
    "plan de prévention des risques naturel",
    "plan de prévention des risques miniers",
    "plan de prévention des risques technologiques",
    "date de publication au journal officelles",
]

for col in cols_order:
  if col not in final_df.columns:
    final_df[col] = np.nan

final_df = final_df[cols_order]

# 8. Envoi direct dans BigQuery
final_df.to_gbq(
    destination_table="sous_nos_pieds.all_risques_catnat",  # Format: dataset.nom_de_table
    project_id="my-first-project-502010",  # Votre ID de projet Google Cloud exact
    if_exists="replace",
    location="EU",
)

print("Table envoyée avec succès dans BigQuery !")

Table envoyée avec succès dans BigQuery !


In [ ]:
final_df

,swi_unif_mens,code commune,code région,code département,nom commune,nom region,nom départment,nb_innondation,nb_tempete,nb_seisme,plan de prévention des risques naturel,plan de prévention des risques miniers,plan de prévention des risques technologiques,date de publication au journal officelles
0,NaN,01004,84.0,01,amberieu-en-bugey,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
1,NaN,01007,84.0,01,ambronay,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1990-03-23
2,0.852557,01010,84.0,01,anglefort,auvergne-rhone-alpes,ain,NaN,NaN,NaN,Oui,NaN,NaN,NaN
3,NaN,01016,84.0,01,arbigny,auvergne-rhone-alpes,ain,4.0,0.0,0.0,Oui,NaN,NaN,1990-12-19
4,NaN,01017,84.0,01,argis,auvergne-rhone-alpes,ain,2.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14240,NaN,97615,6.0,976,pamandzi,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14241,NaN,97616,6.0,976,sada,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14242,NaN,97617,6.0,976,tsingoni,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14243,NaN,97701,7.0,977,saint-barthelemy,saint-barthelemy,saint-barthelemy,NaN,NaN,NaN,Oui,NaN,NaN,NaN


In [ ]:
import numpy as np
import pandas as pd

# 1. Charger tous les fichiers source
catnat = pd.read_csv("catnat_staging.csv", low_memory=False)
pprn = pd.read_csv("pprn_garspar_staging.csv", low_memory=False)
pprm = pd.read_csv("pprm_gaspar_staging.csv", low_memory=False)
pprt = pd.read_csv("pprt_gaspar_staging.csv", low_memory=False)

# Charger swi_par_commune si disponible (assurez-vous d'avoir exporté votre image/donnée en CSV si ce n'est pas fait)
try:
  swi_df = pd.read_csv("swi_par_commune.csv", low_memory=False)
  swi_df["code_commune_clean"] = (
      swi_df["code_commune"].astype(str).str.zfill(5)
  )
  swi_agg = (
      swi_df.groupby("code_commune_clean")["SWI_UNIF_MENS"]
      .mean()
      .reset_index()
  )
  swi_agg = swi_agg.rename(
      columns={
          "code_commune_clean": "code commune",
          "SWI_UNIF_MENS": "swi_unif_mens",
      }
  )
except FileNotFoundError:
  swi_agg = pd.DataFrame(columns=["code commune", "swi_unif_mens"])

# 2. Traitement et agrégation des Catastrophes Naturelles (CatNat)
def classify_risk(lib):
  if pd.isna(lib):
    return None
  lib_lower = str(lib).lower()
  if "inondation" in lib_lower or "coulees de boue" in lib_lower:
    return "nb_innondation"
  elif "tempete" in lib_lower:
    return "nb_tempete"
  elif "sismique" in lib_lower or "seisme" in lib_lower:
    return "nb_seisme"
  return None


catnat["risk_type"] = catnat["lib_risque_jo"].apply(classify_risk)
catnat_counts = (
    catnat.dropna(subset=["risk_type"])
    .pivot_table(
        index="code_commune",
        columns="risk_type",
        values="id_gaspar",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)

for col in ["nb_innondation", "nb_tempete", "nb_seisme"]:
  if col not in catnat_counts.columns:
    catnat_counts[col] = 0

catnat_jo = (
    catnat.groupby("code_commune")["date_publication_jo"]
    .max()
    .reset_index()
)
catnat_jo = catnat_jo.rename(
    columns={"date_publication_jo": "date de publication au journal officelles"}
)
catnat_merged = pd.merge(
    catnat_counts, catnat_jo, on="code_commune", how="outer"
)

# 3. Traitement des Plans de Prévention (PPRn, PPRm, PPRt)
pprn["code_commune_clean"] = pprn["code_commune"].fillna(
    pprn["code_commune_final"]
)
pprn_agg = (
    pprn.dropna(subset=["code_commune_clean"])
    .groupby("code_commune_clean")
    .agg({
        "nom_commune": "first",
        "nom_region": "first",
        "nom_departement": "first",
        "code_insee_region": "first",
        "code_insee_departement": "first",
        "libelle_etat": lambda x: "Oui" if any(pd.notna(x)) else "Non",
    })
    .reset_index()
)

pprn_agg = pprn_agg.rename(
    columns={
        "code_commune_clean": "code commune",
        "code_insee_region": "code région",
        "code_insee_departement": "code département",
        "nom_commune": "nom commune",
        "nom_region": "nom region",
        "nom_departement": "nom départment",
        "libelle_etat": "plan de prévention des risques naturel",
    }
)

pprm["code_commune_clean"] = (
    pprm["CODE INSEE COMMUNE"].astype(str).str.zfill(5)
)
pprm_agg = (
    pprm.groupby("code_commune_clean")
    .agg({
        "LIBELLE ETAT": lambda x: (
            "Oui" if any(pd.notna(x) & (x != "none")) else "Non"
        )
    })
    .reset_index()
    .rename(
        columns={
            "code_commune_clean": "code commune",
            "LIBELLE ETAT": "plan de prévention des risques miniers",
        }
    )
)

pprt["code_commune_clean"] = (
    pprt["CODE INSEE COMMUNE"].astype(str).str.zfill(5)
)
pprt_agg = (
    pprt.groupby("code_commune_clean")
    .agg({
        "LIBELLE ETAT": lambda x: (
            "Oui" if any(pd.notna(x) & (x != "none")) else "Non"
        )
    })
    .reset_index()
    .rename(
        columns={
            "code_commune_clean": "code commune",
            "LIBELLE ETAT": "plan de prévention des risques technologiques",
        }
    )
)

# 4. Fusion finale de toutes les tables
catnat_merged["code commune"] = (
    catnat_merged["code_commune"].astype(str).str.zfill(5)
)

final_df = pprn_agg[
    [
        "code commune",
        "code région",
        "code département",
        "nom commune",
        "nom region",
        "nom départment",
    ]
].copy()

if not swi_agg.empty:
  final_df = pd.merge(final_df, swi_agg, on="code commune", how="left")
else:
  final_df["swi_unif_mens"] = np.nan

final_df = pd.merge(
    final_df,
    catnat_merged[[
        "code commune",
        "nb_innondation",
        "nb_tempete",
        "nb_seisme",
        "date de publication au journal officelles",
    ]],
    on="code commune",
    how="left",
)
final_df = pd.merge(
    final_df,
    pprn_agg[["code commune", "plan de prévention des risques naturel"]],
    on="code commune",
    how="left",
)
final_df = pd.merge(final_df, pprm_agg, on="code commune", how="left")
final_df = pd.merge(final_df, pprt_agg, on="code commune", how="left")

# Ordonnancement strict des colonnes demandées
cols_order = [
    "swi_unif_mens",
    "code commune",
    "code région",
    "code département",
    "nom commune",
    "nom region",
    "nom départment",
    "nb_innondation",
    "nb_tempete",
    "nb_seisme",
    "plan de prévention des risques naturel",
    "plan de prévention des risques miniers",
    "plan de prévention des risques technologiques",
    "date de publication au journal officelles",
]

for col in cols_order:
  if col not in final_df.columns:
    final_df[col] = np.nan

final_df = final_df[cols_order]

# 5. Affichage direct et export
print("Affichage des 10 premières lignes :")
display(final_df.head(10))  # Si vous êtes sous Jupyter/Colab

# Sauvegarde locale en CSV pour vérification
final_df.to_csv("table_finale_risques.csv", index=False, encoding="utf-8-sig")

# 6. Envoi vers BigQuery (avec la bonne syntaxe dataset.table)
final_df.to_gbq(
    destination_table="intermediate.all_risques_catnat",
    project_id="sous-nos-pieds",
    if_exists="replace",
    location="EU",
)
print("Table exportée avec succès vers BigQuery !")

Affichage des 10 premières lignes :


,swi_unif_mens,code commune,code région,code département,nom commune,nom region,nom départment,nb_innondation,nb_tempete,nb_seisme,plan de prévention des risques naturel,plan de prévention des risques miniers,plan de prévention des risques technologiques,date de publication au journal officelles
0,NaN,01004,84.0,01,amberieu-en-bugey,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
1,NaN,01007,84.0,01,ambronay,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1990-03-23
2,0.852557,01010,84.0,01,anglefort,auvergne-rhone-alpes,ain,NaN,NaN,NaN,Oui,NaN,NaN,NaN
3,NaN,01016,84.0,01,arbigny,auvergne-rhone-alpes,ain,4.0,0.0,0.0,Oui,NaN,NaN,1990-12-19
4,NaN,01017,84.0,01,argis,auvergne-rhone-alpes,ain,2.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
5,NaN,01022,84.0,01,artemare,auvergne-rhone-alpes,ain,1.0,0.0,0.0,Oui,NaN,NaN,1990-03-23
6,NaN,01023,84.0,01,asnieres-sur-saone,auvergne-rhone-alpes,ain,5.0,0.0,0.0,Oui,NaN,NaN,1990-12-19
7,NaN,01027,84.0,01,balan,auvergne-rhone-alpes,ain,2.0,0.0,0.0,Oui,NaN,Oui,1990-03-23
8,NaN,01030,84.0,01,beauregard,auvergne-rhone-alpes,ain,2.0,0.0,0.0,Oui,NaN,NaN,1983-09-11
9,NaN,01031,84.0,01,bellignat,auvergne-rhone-alpes,ain,1.0,0.0,0.0,Oui,NaN,NaN,1990-03-23


Table exportée avec succès vers BigQuery !


In [ ]:
# Renommer proprement les colonnes pour éviter les erreurs Looker Studio / BigQuery
final_df = final_df.rename(
    columns={
        "code commune": "code_commune",
        "code région": "code_region",
        "code département": "code_departement",
        "nom commune": "nom_commune",
        "nom region": "nom_region",
        "nom départment": "nom_departement",
        "date de publication au journal officelles": "date_publication_jo",
        "plan de prévention des risques naturel": "ppr_naturel",
        "plan de prévention des risques miniers": "ppr_minier",
        "plan de prévention des risques technologiques": "ppr_technologique",
    }
)

# Ré-envoyer la table propre dans BigQuery
final_df.to_gbq(
    destination_table="intermediate.all_risques_catnat",
    project_id="sous-nos-pieds",
    if_exists="replace",
    location="EU",
)
print("Table mise à jour avec des noms propres !")

Table mise à jour avec des noms propres !


In [ ]:
final_df

,swi_unif_mens,code_commune,code_region,code_departement,nom_commune,nom_region,nom_departement,nb_innondation,nb_tempete,nb_seisme,ppr_naturel,ppr_minier,ppr_technologique,date_publication_jo
0,NaN,01004,84.0,01,amberieu-en-bugey,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
1,NaN,01007,84.0,01,ambronay,auvergne-rhone-alpes,ain,3.0,0.0,0.0,Oui,NaN,NaN,1990-03-23
2,0.852557,01010,84.0,01,anglefort,auvergne-rhone-alpes,ain,NaN,NaN,NaN,Oui,NaN,NaN,NaN
3,NaN,01016,84.0,01,arbigny,auvergne-rhone-alpes,ain,4.0,0.0,0.0,Oui,NaN,NaN,1990-12-19
4,NaN,01017,84.0,01,argis,auvergne-rhone-alpes,ain,2.0,0.0,0.0,Oui,NaN,NaN,1992-03-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14240,NaN,97615,6.0,976,pamandzi,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14241,NaN,97616,6.0,976,sada,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14242,NaN,97617,6.0,976,tsingoni,mayotte,mayotte,NaN,NaN,NaN,Oui,NaN,NaN,NaN
14243,NaN,97701,7.0,977,saint-barthelemy,saint-barthelemy,saint-barthelemy,NaN,NaN,NaN,Oui,NaN,NaN,NaN


In [ ]:
# 6. Envoi vers BigQuery (avec la bonne syntaxe dataset.table)
final_df.to_gbq(
    destination_table="sous_nos_pieds.all_risques_catnat",
    project_id="my-first-project-502010",
    if_exists="replace",
    location="EU",
)
print("Table exportée avec succès vers BigQuery !")

Table exportée avec succès vers BigQuery !
